In [ ]:
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score , classification_report, confusion_matrix
from sklearn.model_selection import GridSearchCV
import mlflow
from pathlib import Path


import matplotlib.pyplot as plt
import seaborn as sns

# Load datasets

In [ ]:
TRAIN_PATH = "./../data/processed/train.csv"
VALIDATION_PATH = "./../data/processed/validation.csv"
TEST_PATH = "./../data/processed/test.csv"

In [ ]:
train_df= pd.read_csv(TRAIN_PATH)
validation_df= pd.read_csv(VALIDATION_PATH)
test_df= pd.read_csv(TEST_PATH)

X_train = train_df.drop(columns=["target"])
y_train = train_df["target"]

X_validation = validation_df.drop(columns=["target"])
y_validation = validation_df["target"]

X_test = test_df.drop(columns=["target"])
y_test = test_df["target"]

del train_df, validation_df, test_df

In [ ]:
print("Training set shape:", X_train.shape)
print("Validation set shape:", X_validation.shape)
print("Test set shape:", X_test.shape)

In [ ]:
print(X_train.isna().sum().sum())
print(X_validation.isna().sum().sum())
print(X_test.isna().sum().sum())

In [ ]:
print(y_train.isna().sum().sum())
print(y_validation.isna().sum().sum())
print(y_test.isna().sum().sum())

# MLFLOW

In [ ]:
EXPERIMENT_NAME="house_price_prediction_experiment"
TRAIN_CONFUSION_MATRIX_PATH = './../reports/train_confusion_matrix.png'
VALIDATION_CONFUSION_MATRIX_PATH = './../reports/validation_confusion_matrix.png'
TEST_CONFUSION_MATRIX_PATH = './../reports/test_confusion_matrix.png'
MLFLOW_DB_MODEL_PATH = './../models/mlflow.db'

In [ ]:
mlflow.set_tracking_uri(f"sqlite:///{MLFLOW_DB_MODEL_PATH}")
mlflow.set_experiment(EXPERIMENT_NAME)
experiment = mlflow.get_experiment_by_name(EXPERIMENT_NAME)
experiment_id = experiment.experiment_id

print("EXPERIMENT INFO:")
print(f"Name: {experiment.name}")
print(f"ID: {experiment.experiment_id}")
print(f"Artifact Location: {experiment.artifact_location}")
print(f"Tags: {experiment.tags}")
print(f"Lifecycle Stage: {experiment.lifecycle_stage}")
print(f"Creation timestamp: {experiment.creation_time}")

# classification evaluation

In [ ]:
def save_confusion_matrix(cm, filename):
    plt.figure()
    sns.heatmap(cm, annot=True, fmt="d") 
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.savefig(filename)
    plt.close()

In [ ]:
def evaluate_model(y_pred, y_true):
    acc = accuracy_score(y_true, y_pred)
    report = classification_report(y_true, y_pred)
    cm = confusion_matrix(y_true, y_pred)
    return acc, report, cm

# Model training

### logistic regression

In [ ]:
with mlflow.start_run(run_name='logistic_regression_v1', experiment_id=experiment_id):
    params = {
        'penalty': 'l2',         
        'C': 1.0,                
        'solver': 'lbfgs',       
        'max_iter': 10000,        
        'random_state': 42
    }

    logreg = LogisticRegression(**params)
    logreg.fit(X_train, y_train)

    y_train_pred = logreg.predict(X_train)
    y_validation_pred = logreg.predict(X_validation)

    train_acc, train_report, train_cm = evaluate_model(y_train_pred, y_train)
    val_acc, val_report, val_cm = evaluate_model(y_validation_pred, y_validation)

    print("Training Accuracy:", train_acc)
    print("Validation Accuracy:", val_acc)
    print("Training Classification Report:\n", train_report)
    print("Validation Classification Report:\n", val_report)

    save_confusion_matrix(train_cm, TRAIN_CONFUSION_MATRIX_PATH)
    save_confusion_matrix(val_cm, VALIDATION_CONFUSION_MATRIX_PATH)

    mlflow.log_params(params)
    mlflow.log_metrics({
        'train_acc': train_acc,
        'val_acc': val_acc
    })
    mlflow.log_text(train_report, 'train_classification_report.txt')
    mlflow.log_text(val_report, 'validation_classification_report.txt')
    mlflow.log_artifact(TRAIN_CONFUSION_MATRIX_PATH)
    mlflow.log_artifact(VALIDATION_CONFUSION_MATRIX_PATH)

    mlflow.sklearn.log_model(logreg, 'logistic_regression_model')

In [ ]:
print("Training Accuracy:", train_acc)
print("Validation Accuracy:", val_acc)
print("Training Classification Report:\n", train_report)
print("Validation Classification Report:\n", val_report)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV
import mlflow
import mlflow.sklearn

with mlflow.start_run(run_name='logistic_regression_tuned_v1', experiment_id=experiment_id):

    base_model = LogisticRegression(max_iter=10000, random_state=42)

    param_grid = {
        "penalty": ["l2"],
        "C": [0.01, 0.1, 1, 10],
        "solver": ["lbfgs"]
    }

    grid_search = GridSearchCV(
        estimator=base_model,
        param_grid=param_grid,
        cv=5,
        scoring="f1_macro",
        n_jobs=-1
    )

    grid_search.fit(X_train, y_train)

    best_model = grid_search.best_estimator_
    best_params = grid_search.best_params_

    y_train_pred = best_model.predict(X_train)
    y_validation_pred = best_model.predict(X_validation)

    train_acc, train_report, train_cm = evaluate_model(y_train_pred, y_train)
    val_acc, val_report, val_cm = evaluate_model(y_validation_pred, y_validation)

    print("Best Params:", best_params)
    print("Training Accuracy:", train_acc)
    print("Validation Accuracy:", val_acc)
    print("Training Classification Report:\n", train_report)
    print("Validation Classification Report:\n", val_report)

    save_confusion_matrix(train_cm, TRAIN_CONFUSION_MATRIX_PATH)
    save_confusion_matrix(val_cm, VALIDATION_CONFUSION_MATRIX_PATH)

    mlflow.log_params(best_params)
    mlflow.log_metrics({
        'train_acc': train_acc,
        'val_acc': val_acc,
        'cv_best_score': grid_search.best_score_
    })

    mlflow.log_text(train_report, 'train_classification_report.txt')
    mlflow.log_text(val_report, 'validation_classification_report.txt')

    mlflow.log_artifact(TRAIN_CONFUSION_MATRIX_PATH)
    mlflow.log_artifact(VALIDATION_CONFUSION_MATRIX_PATH)

    mlflow.sklearn.log_model(best_model, 'logistic_regression_model')

### decision tree

In [ ]:
with mlflow.start_run(run_name='decision_tree_v1', experiment_id=experiment_id):
    params = {
        'criterion': 'gini',       
        'max_depth': 20,           
        'min_samples_split': 5,    
        'min_samples_leaf': 2,     
        'max_features': None,      
        'random_state': 42
    }

    dtc = DecisionTreeClassifier(**params)
    dtc.fit(X_train, y_train)

    y_train_pred = dtc.predict(X_train)
    y_validation_pred = dtc.predict(X_validation)

    train_acc, train_report, train_cm = evaluate_model(y_train_pred, y_train)
    val_acc, val_report, val_cm = evaluate_model(y_validation_pred, y_validation)

    save_confusion_matrix(train_cm, TRAIN_CONFUSION_MATRIX_PATH)
    save_confusion_matrix(val_cm, VALIDATION_CONFUSION_MATRIX_PATH)

    mlflow.log_params(params)
    mlflow.log_metrics({
        'train_acc': train_acc,
        'val_acc': val_acc
    })

    mlflow.log_text(train_report, 'train_classification_report.txt')
    mlflow.log_text(val_report, 'validation_classification_report.txt')

    mlflow.log_artifact(TRAIN_CONFUSION_MATRIX_PATH)
    mlflow.log_artifact(VALIDATION_CONFUSION_MATRIX_PATH)

    mlflow.sklearn.log_model(dtc, 'decision_tree_model')

In [ ]:
print("Training Accuracy:", train_acc)
print("Validation Accuracy:", val_acc)
print("Training Classification Report:\n", train_report)
print("Validation Classification Report:\n", val_report)

In [ ]:
with mlflow.start_run(run_name='decision_tree_tuned_v1', experiment_id=experiment_id):

    base_model = DecisionTreeClassifier(random_state=42)

    param_grid = {
        "max_depth": [5, 10, 20, None],
        "min_samples_split": [2, 5, 10]

    }

    grid_search = GridSearchCV(
        estimator=base_model,
        param_grid=param_grid,
        cv=5,
        scoring="f1_macro",
        n_jobs=-1
    )

    grid_search.fit(X_train, y_train)

    best_model = grid_search.best_estimator_
    best_params = grid_search.best_params_

    y_train_pred = best_model.predict(X_train)
    y_validation_pred = best_model.predict(X_validation)

    train_acc, train_report, train_cm = evaluate_model(y_train_pred, y_train)
    val_acc, val_report, val_cm = evaluate_model(y_validation_pred, y_validation)

    print("Best Params:", best_params)
    print("Training Accuracy:", train_acc)
    print("Validation Accuracy:", val_acc)
    print("CV Best Score:", grid_search.best_score_)

    save_confusion_matrix(train_cm, TRAIN_CONFUSION_MATRIX_PATH)
    save_confusion_matrix(val_cm, VALIDATION_CONFUSION_MATRIX_PATH)

    mlflow.log_params(best_params)

    mlflow.log_metrics({
        'train_acc': train_acc,
        'val_acc': val_acc,
        'cv_best_score': grid_search.best_score_
    })

    mlflow.log_text(train_report, 'train_classification_report.txt')
    mlflow.log_text(val_report, 'validation_classification_report.txt')

    mlflow.log_artifact(TRAIN_CONFUSION_MATRIX_PATH)
    mlflow.log_artifact(VALIDATION_CONFUSION_MATRIX_PATH)

    mlflow.sklearn.log_model(best_model, 'decision_tree_model')

In [ ]:
print("Training Accuracy:", train_acc)
print("Validation Accuracy:", val_acc)
print("Training Classification Report:\n", train_report)
print("Validation Classification Report:\n", val_report)

### Random forest

In [ ]:
with mlflow.start_run(run_name='random_forest_v1', experiment_id=experiment_id):
    params = {
    'n_estimators': 300,          
    'max_depth': 20,             
    'min_samples_split': 5,      
    'min_samples_leaf': 2,        
    'max_features': 'sqrt',       
    'bootstrap': True,
    'random_state': 42,
    'n_jobs': -1                 
    }

    rfc = RandomForestClassifier(**params)
    rfc.fit(X_train, y_train)
    y_train_pred = rfc.predict(X_train)
    y_validation_pred = rfc.predict(X_validation)

    train_acc, train_report, train_cm = evaluate_model(y_train_pred, y_train)
    val_acc, val_report, val_cm = evaluate_model(y_validation_pred, y_validation)

    save_confusion_matrix(train_cm, TRAIN_CONFUSION_MATRIX_PATH)
    save_confusion_matrix(val_cm, VALIDATION_CONFUSION_MATRIX_PATH)

    mlflow.log_params(params)
    mlflow.log_metrics({
        'train_acc': train_acc,
        'val_acc': val_acc
    })

    mlflow.log_text(train_report, 'train_classification_report.txt')
    mlflow.log_text(val_report, 'validation_classification_report.txt')

    mlflow.log_artifact(TRAIN_CONFUSION_MATRIX_PATH)
    mlflow.log_artifact(VALIDATION_CONFUSION_MATRIX_PATH)

    mlflow.sklearn.log_model(rfc, 'random_forest_model')

In [ ]:
print("Training Accuracy:", train_acc)
print("Validation Accuracy:", val_acc)
print("Training Classification Report:\n", train_report)
print("Validation Classification Report:\n", val_report)

In [ ]:
with mlflow.start_run(run_name='random_forest_tuned_v1', experiment_id=experiment_id):

    base_model = RandomForestClassifier(random_state=42, n_jobs=-1)

    param_grid = {
    
        "n_estimators": [100, 300],
        "max_depth": [10, 20, None]
  
    }

    grid_search = GridSearchCV(
        estimator=base_model,
        param_grid=param_grid,
        cv=5,
        scoring="f1_macro",
        n_jobs=-1
    )

    grid_search.fit(X_train, y_train)

    best_model = grid_search.best_estimator_
    best_params = grid_search.best_params_

    y_train_pred = best_model.predict(X_train)
    y_validation_pred = best_model.predict(X_validation)

    train_acc, train_report, train_cm = evaluate_model(y_train_pred, y_train)
    val_acc, val_report, val_cm = evaluate_model(y_validation_pred, y_validation)

    print("Best Params:", best_params)
    print("Training Accuracy:", train_acc)
    print("Validation Accuracy:", val_acc)
    print("CV Best Score:", grid_search.best_score_)

    save_confusion_matrix(train_cm, TRAIN_CONFUSION_MATRIX_PATH)
    save_confusion_matrix(val_cm, VALIDATION_CONFUSION_MATRIX_PATH)

    mlflow.log_params(best_params)

    mlflow.log_metrics({
        'train_acc': train_acc,
        'val_acc': val_acc,
        'cv_best_score': grid_search.best_score_
    })

    mlflow.log_text(train_report, 'train_classification_report.txt')
    mlflow.log_text(val_report, 'validation_classification_report.txt')

    mlflow.log_artifact(TRAIN_CONFUSION_MATRIX_PATH)
    mlflow.log_artifact(VALIDATION_CONFUSION_MATRIX_PATH)

    mlflow.sklearn.log_model(best_model, 'random_forest_model')

### XGBoost

In [ ]:
with mlflow.start_run(run_name='xgboost_v1', experiment_id=experiment_id):
    params = {
        'n_estimators': 300,        
        'max_depth': 6,             
        'learning_rate': 0.1,       
        'subsample': 0.8,           
        'colsample_bytree': 0.8,    
        'gamma': 0,                 
        'reg_alpha': 0,             
        'reg_lambda': 1,            
        'random_state': 42,
        'n_jobs': -1,
    }

    xgb = XGBClassifier(**params)
    xgb.fit(X_train, y_train)

    # Predictions
    y_train_pred = xgb.predict(X_train)
    y_validation_pred = xgb.predict(X_validation)

    # Evaluation
    train_acc, train_report, train_cm = evaluate_model(y_train_pred, y_train)
    val_acc, val_report, val_cm = evaluate_model(y_validation_pred, y_validation)

    # Save confusion matrices as images
    save_confusion_matrix(train_cm, TRAIN_CONFUSION_MATRIX_PATH)
    save_confusion_matrix(val_cm, VALIDATION_CONFUSION_MATRIX_PATH)

    # Logging
    mlflow.log_params(params)
    mlflow.log_metrics({
        'train_acc': train_acc,
        'val_acc': val_acc
    })

    mlflow.log_text(train_report, 'train_classification_report.txt')
    mlflow.log_text(val_report, 'validation_classification_report.txt')

    mlflow.log_artifact(TRAIN_CONFUSION_MATRIX_PATH)
    mlflow.log_artifact(VALIDATION_CONFUSION_MATRIX_PATH)

    mlflow.sklearn.log_model(xgb, 'xgboost_model')

In [ ]:
print("Training Accuracy:", train_acc)
print("Validation Accuracy:", val_acc)
print("Training Classification Report:\n", train_report)
print("Validation Classification Report:\n", val_report)

In [ ]:
with mlflow.start_run(run_name='xgboost_tuned_v1', experiment_id=experiment_id):
    base_model = XGBClassifier(
        random_state=42,
        n_jobs=-1,
        eval_metric='logloss'
    )

    param_grid = {
        "n_estimators": [200, 300],
        "max_depth": [3, 5],
        "learning_rate": [0.05, 0.1]
    }

    grid_search = GridSearchCV(
        estimator=base_model,
        param_grid=param_grid,
        cv=3,
        scoring='accuracy',
        n_jobs=-1,
        verbose=1
    )

    grid_search.fit(X_train, y_train)

    best_model = grid_search.best_estimator_
    best_params = grid_search.best_params_

    y_train_pred = best_model.predict(X_train)
    y_validation_pred = best_model.predict(X_validation)

    train_acc, train_report, train_cm = evaluate_model(y_train_pred, y_train)
    val_acc, val_report, val_cm = evaluate_model(y_validation_pred, y_validation)

    print("Best Params:", best_params)
    print("Training Accuracy:", train_acc)
    print("Validation Accuracy:", val_acc)
    print("CV Best Score:", grid_search.best_score_)

    save_confusion_matrix(train_cm, TRAIN_CONFUSION_MATRIX_PATH)
    save_confusion_matrix(val_cm, VALIDATION_CONFUSION_MATRIX_PATH)

    mlflow.log_params(best_params)

    mlflow.log_metrics({
        'train_acc': train_acc,
        'val_acc': val_acc,
        'cv_best_score': grid_search.best_score_
    })

    mlflow.log_text(train_report, 'train_classification_report.txt')
    mlflow.log_text(val_report, 'validation_classification_report.txt')

    mlflow.log_artifact(TRAIN_CONFUSION_MATRIX_PATH)
    mlflow.log_artifact(VALIDATION_CONFUSION_MATRIX_PATH)

    mlflow.sklearn.log_model(best_model, 'xgboost_model')

In [ ]:
print("Training Accuracy:", train_acc)
print("Validation Accuracy:", val_acc)
print("Training Classification Report:\n", train_report)
print("Validation Classification Report:\n", val_report)

# TESTING

In [ ]:
# # Load model
# run_id = 'a74aa6166c014c009699b9c81c05918e'
# model_name = 'random_forest_classifier'
# model_uri = f'runs:/{run_id}/{model_name}'
# rfc = mlflow.sklearn.load_model(model_uri=model_uri)

# # Predict
# y_pred = rfc.predict(X_test)
# y_pred = pd.DataFrame(y_pred, columns=['prediction'])

# eval_acc, eval_report, eval_cm = evaluate_model(y_pred, y_test)
# print("Test Accuracy:", eval_acc)
# print("Classification Report:\n", eval_report)
# save_confusion_matrix(eval_cm, "test_confusion_matrix.png")

### SVC

In [ ]:
with mlflow.start_run(run_name='svc_v1', experiment_id=experiment_id):
    params = {
        'C': 1.0,                
        'kernel': 'rbf',        
        'gamma': 'scale',       
        'random_state': 42
    }

    svc = SVC(**params)
    svc.fit(X_train, y_train)

    y_train_pred = svc.predict(X_train)
    y_validation_pred = svc.predict(X_validation)

    train_acc, train_report, train_cm = evaluate_model(y_train_pred, y_train)
    val_acc, val_report, val_cm = evaluate_model(y_validation_pred, y_validation)

    save_confusion_matrix(train_cm, TRAIN_CONFUSION_MATRIX_PATH)
    save_confusion_matrix(val_cm, VALIDATION_CONFUSION_MATRIX_PATH)

    mlflow.log_params(params)
    mlflow.log_metrics({
        'train_acc': train_acc,
        'val_acc': val_acc
    })

    mlflow.log_text(train_report, 'train_classification_report.txt')
    mlflow.log_text(val_report, 'validation_classification_report.txt')

    mlflow.log_artifact(TRAIN_CONFUSION_MATRIX_PATH)
    mlflow.log_artifact(VALIDATION_CONFUSION_MATRIX_PATH)

    mlflow.sklearn.log_model(svc, 'svc_model')

In [ ]:
print("Training Accuracy:", train_acc)
print("Validation Accuracy:", val_acc)
print("Training Classification Report:\n", train_report)
print("Validation Classification Report:\n", val_report)

In [ ]:
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV
import mlflow
import mlflow.sklearn

with mlflow.start_run(run_name='svc_tuned_v1', experiment_id=experiment_id):

    base_model = SVC()

    param_grid = {
        "C": [0.1, 1, 10],
        "kernel": ["rbf"],
        "gamma": ["scale", "auto"]
    }

    grid_search = GridSearchCV(
        estimator=base_model,
        param_grid=param_grid,
        cv=5,
        scoring='accuracy',
        n_jobs=-1
    )

    grid_search.fit(X_train, y_train)

    best_model = grid_search.best_estimator_
    best_params = grid_search.best_params_

    y_train_pred = best_model.predict(X_train)
    y_validation_pred = best_model.predict(X_validation)

    train_acc, train_report, train_cm = evaluate_model(y_train_pred, y_train)
    val_acc, val_report, val_cm = evaluate_model(y_validation_pred, y_validation)

    print("Best Params:", best_params)
    print("Training Accuracy:", train_acc)
    print("Validation Accuracy:", val_acc)

    save_confusion_matrix(train_cm, TRAIN_CONFUSION_MATRIX_PATH)
    save_confusion_matrix(val_cm, VALIDATION_CONFUSION_MATRIX_PATH)

    mlflow.log_params(best_params)

    mlflow.log_metrics({
        'train_acc': train_acc,
        'val_acc': val_acc,
        'cv_best_score': grid_search.best_score_
    })

    mlflow.log_text(train_report, 'train_classification_report.txt')
    mlflow.log_text(val_report, 'validation_classification_report.txt')

    mlflow.log_artifact(TRAIN_CONFUSION_MATRIX_PATH)
    mlflow.log_artifact(VALIDATION_CONFUSION_MATRIX_PATH)

    mlflow.sklearn.log_model(best_model, 'svc_model')

In [ ]:
print("Training Accuracy:", train_acc)
print("Validation Accuracy:", val_acc)
print("Training Classification Report:\n", train_report)
print("Validation Classification Report:\n", val_report)